In [3]:
import re, urllib.request
from math import gcd
from functools import reduce

URL_MINUS = "https://homes.cerias.purdue.edu/~ssw/cun/pmain126.txt"

# ── fetch & parse both Table 2- and Table 2+ ─────────────────

def fetch(url):
    with urllib.request.urlopen(url) as r:
        return r.read().decode("ascii", errors="replace")

def parse_block(text, header):
    """Extract one table block starting at 'header' and parse its entries."""
    start = text.find(header)
    if start == -1:
        return {}
    next_table = text.find("Table", start + len(header))
    chunk = text[start : next_table if next_table != -1 else None]

    result = {}
    entry_re = re.compile(r'^\s{0,6}(\d+)\s+(.*)')
    cont_re  = re.compile(r'^\s{20,}(.*)')
    current_n, current_tail = None, ""

    def flush(n, tail):
        if n is None:
            return
        alg_raw = re.findall(r'\(([^)]+)\)', tail)
        algebraic = []
        for a in alg_raw:
            algebraic += [int(x) for x in a.split(',') if x.strip().isdigit()]
        tail2 = re.sub(r'\([^)]*\)', '', tail).strip()
        tokens = [t.strip().rstrip('*') for t in tail2.split('.') if t.strip()]
        primes, complete = [], True
        for tok in tokens:
            if re.fullmatch(r'\d+', tok):
                primes.append(int(tok))
            elif re.fullmatch(r'[Pp]\d+', tok):
                complete = False
        result[n] = {"primes": primes, "algebraic": algebraic, "complete": complete}

    for line in chunk.splitlines():
        m = entry_re.match(line)
        if m:
            flush(current_n, current_tail)
            current_n, current_tail = int(m.group(1)), m.group(2)
        else:
            c = cont_re.match(line)
            if c and current_n is not None:
                current_tail += c.group(1)
    flush(current_n, current_tail)
    return result

def load_tables():
    text = fetch(URL_MINUS)
    minus = parse_block(text, "Table 2-")   # 2^n - 1, n odd
    plus  = parse_block(text, "Table 2+")   # 2^n + 1, n odd
    return minus, plus

# ── cyclotomic decomposition ──────────────────────────────────

from sympy import divisors, isprime
from sympy.ntheory import factorint as sym_factorint
from sympy.functions.combinatorial.numbers import mobius

def phi2(n):
    """
    Evaluate Φ_n(2) — the n-th cyclotomic polynomial at x=2.
    Φ_n(2) = ∏_{d|n} (2^d - 1)^μ(n/d)   (Möbius inversion)

    Computed as numerator / denominator to avoid premature integer division.
    """
    num = 1
    den = 1
    for d in divisors(n):
        mu = mobius(n // d)
        if mu == 1:
            num *= (2**d - 1)
        elif mu == -1:
            den *= (2**d - 1)
    return num // den

def sympy_factor(n):
    """Factor n using sympy. Returns (primes, complete=True)."""
    fdict = sym_factorint(int(n))
    return sorted(fdict.keys()), True

def primitive_prime_factors(k, minus_table, plus_table):
    """
    Return all known prime factors of Φ_k(2) (the primitive part of 2^k - 1)
    together with a completeness flag.
    """
    # Factorise k = 2^a * m, m odd
    a = 0
    m = k
    while m % 2 == 0:
        a += 1
        m //= 2

    if a == 0:
        # k odd: unlabelled primes in Table 2- are the primitive ones
        if k not in minus_table:
            return None, False
        entry = minus_table[k]
        return entry["primes"], entry["complete"]

    elif a == 1:
        # k = 2*m, m odd: Φ_k(2) = 2^m + 1, listed in Table 2+
        if m not in plus_table:
            return None, False
        entry = plus_table[m]
        return entry["primes"], entry["complete"]

    else:
        # k = 2^a * m, a >= 2: compute Φ_k(2) and factor it.
        val = phi2(k)
        # Try FactorDB first (instant for known factorizations)
        primes, complete = factordb_factors(val)
        if complete:
            return primes, True
        # Fall back to sympy for small numbers only
        if val.bit_length() <= 100:
            return sympy_factor(val)
        return primes, complete

# ── FactorDB fallback ─────────────────────────────────────────

import json, urllib.parse

def factordb_factors(n):
    try:
        url = f"http://factordb.com/api?query={n}"
        with urllib.request.urlopen(url, timeout=10) as r:
            data = json.load(r)
        status = data.get("status", "U")
        if status in ("FF", "P", "Prp"):
            primes = [int(p) for p, _ in data["factors"]]
            return primes, True
        elif status == "CF":
            primes = [int(p) for p, _ in data["factors"] if _]
            return primes, False
    except Exception:
        pass
    return [], False

# ── verification ──────────────────────────────────────────────

def verify_factorization(k, primes, complete):
    """
    Verify that the returned prime factors are correct.
    Checks:
      1. Each factor divides Φ_k(2)
      2. Each factor is prime
      3. If complete, the product of all factors (with multiplicity) equals Φ_k(2)
    Returns (ok, message).
    """
    if primes is None:
        return False, "no data"

    val = phi2(k)
    errors = []

    for p in primes:
        if val % p != 0:
            errors.append(f"{p} does not divide Φ_{k}(2)")
        if not isprime(p):
            errors.append(f"{p} is not prime")

    if complete:
        remaining = val
        for p in primes:
            while remaining % p == 0:
                remaining //= p
        if remaining != 1:
            errors.append(f"incomplete: cofactor {remaining} remains")

    if errors:
        return False, "; ".join(errors)
    return True, "ok"

# ── top-level entry point ─────────────────────────────────────

def get_primitive_factors(k, minus_table, plus_table):
    """
    Returns (primes, complete) for Φ_k(2).
    complete=False means a large cofactor is still unknown —
    NTL's IsPrimitive cannot give a full certificate.
    """
    primes, complete = primitive_prime_factors(k, minus_table, plus_table)
    return primes, complete


# ── usage ─────────────────────────────────────────────────────

minus, plus = load_tables()

for k in range(1,1000):
    primes, complete = get_primitive_factors(k, minus, plus)
    ok, msg = verify_factorization(k, primes, complete)
    status = "✓" if ok else f"✗ {msg}"
    print(f"k={k:4d}  complete={complete!s:>5}  Φ_k(2) factors: {primes}  {status}")

k=   1  complete=False  Φ_k(2) factors: None  ✗ no data
k=   2  complete= True  Φ_k(2) factors: [3]  ✓
k=   3  complete= True  Φ_k(2) factors: [7]  ✓
k=   4  complete= True  Φ_k(2) factors: [5]  ✓
k=   5  complete= True  Φ_k(2) factors: [31]  ✓
k=   6  complete= True  Φ_k(2) factors: [3]  ✓
k=   7  complete= True  Φ_k(2) factors: [127]  ✓
k=   8  complete= True  Φ_k(2) factors: [17]  ✓
k=   9  complete= True  Φ_k(2) factors: [73]  ✓
k=  10  complete= True  Φ_k(2) factors: [11]  ✓
k=  11  complete= True  Φ_k(2) factors: [23, 89]  ✓
k=  12  complete= True  Φ_k(2) factors: [13]  ✓
k=  13  complete= True  Φ_k(2) factors: [8191]  ✓
k=  14  complete= True  Φ_k(2) factors: [43]  ✓
k=  15  complete= True  Φ_k(2) factors: [151]  ✓
k=  16  complete= True  Φ_k(2) factors: [257]  ✓
k=  17  complete= True  Φ_k(2) factors: [131071]  ✓
k=  18  complete= True  Φ_k(2) factors: [3, 19]  ✓
k=  19  complete= True  Φ_k(2) factors: [524287]  ✓
k=  20  complete= True  Φ_k(2) factors: [5, 41]  ✓
k=  21  compl

In [10]:
text

'    Table 2-              Factorizations of 2^n-1, n odd, n<1500\n\n    n                                Prime Factors\n    3   7\n    5   31\n    7   127\n    9  (3) 73\n   11   23.89\n   13   8191\n   15  (3,5) 151\n   17   131071\n   19   524287\n   21  (3,7) 7*.337\n   23   47.178481\n   25  (5) 601.1801\n   27  (3,9) 262657\n   29   233.1103.2089\n   31   2147483647\n   33  (3,11) 599479\n   35  (5,7) 71.122921\n   37   223.616318177\n   39  (3,13) 79.121369\n   41   13367.164511353\n   43   431.9719.2099863\n   45  (3,5,9,15) 631.23311\n   47   2351.4513.13264529\n   49  (7) 4432676798593\n   51  (3,17) 103.2143.11119\n   53   6361.69431.20394401\n   55  (5,11) 881.3191.201961\n   57  (3,19) 32377.1212847\n   59   179951.3203431780337\n   61   2305843009213693951\n   63  (3,7,9,21) 92737.649657\n   65  (5,13) 145295143558111\n   67   193707721.761838257287\n   69  (3,23) 10052678938039\n   71   228479.48544121.212885833\n   73   439.2298041.9361973132609\n   75  (3,5,15,25) 1008